In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install meteostat

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 92.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.5/506.5 kB 33.0 MB/s eta 0:00:00
  Attempting uninstall: pytz
    Found existing installation: pytz 2025.2
    Uninstalling pytz-2025.2:
      Successfully uninstalled pytz-2025.2
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.3 which is incompatible.
gradio 5.50.0 requires pandas<3.0,>=1.0, but you have pandas 3.0.3 which is incompatible.


In [3]:
from datetime import date
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import meteostat as ms

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/ereznaamansapienza/data/refs/heads/master/phenology.csv')

In [ ]:
df['Event'].values.unique()

<ArrowStringArray>
[     'First flowering',       'First recorded',            'First cut',
             'Last cut',  'Recorded all winter',     'First ripe fruit',
             'Budburst',           'First leaf', 'First autumn tinting',
  'Full autumn tinting', 'First leaves falling',            'Bare tree']
Length: 12, dtype: str

In [ ]:
df['LongitudeBand'].unique()

<ArrowStringArray>
[ '-3 to -2',    '0 to 1',  '-5 to -4',  '-2 to -1',   '-1 to 0',    '1 to 2',
  '-8 to -7',  '-4 to -3',  '-6 to -5',  '-7 to -6', '-10 to -9',  '-9 to -8',
      '<-11']
Length: 13, dtype: str

In [ ]:
df['LatitudeBand'].unique()

<ArrowStringArray>
['57-58', '51-52', '50-51', '52-53', '53-54', '55-56', '54-55', '56-57',
   '<50', '58-59', '60-61', '59-60']
Length: 12, dtype: str

In [ ]:
df['Longitude'].min()

-10.110906

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
"""
Integrate Woodland Trust phenology observations with Open-Meteo ERA5
historical weather, producing one unified DataFrame ready for loading
into a star schema.

Weather is fetched at the CENTROID of each LatitudeBand × LongitudeBand cell.
With 12 unique lat-bands and 13 lon-bands, this is at most 156 API calls for
the entire dataset, and gives each location dimension member a single,
consistent weather series — cleaner for OLAP than per-recorder GPS points.

Open-Meteo Historical Weather API: https://open-meteo.com/en/docs/historical-weather-api
No API key required.
"""

import os
import time

import requests
import pandas as pd

# --------------------------------------------------------------------------
# CONFIG
# --------------------------------------------------------------------------
PHENOLOGY_CSV = "https://raw.githubusercontent.com/ereznaamansapienza/data/refs/heads/master/phenology.csv"
OUTPUT_CSV    = "phenology_weather_unified.csv"
CACHE_DIR     = "/content/drive/MyDrive/phenology_weather_cache/weather_cache"

# Daily ERA5 variables to pull.  Extend as needed.
DAILY_VARS = [
    "temperature_2m_mean",
    "temperature_2m_max",
    "temperature_2m_min",
    "precipitation_sum",
    "et0_fao_evapotranspiration",   # drought stress proxy
]

WINDOW_DAYS   = 60      # look-back window before each event for "recent" weather
GDD_BASE      = 5.0     # base temp (°C) for growing degree day accumulation
REQUEST_PAUSE = 8.0     # seconds between API calls (~15/min, safely under free limit)

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"

os.makedirs(CACHE_DIR, exist_ok=True)


# --------------------------------------------------------------------------
# BAND CENTROID PARSING
# --------------------------------------------------------------------------

def lat_band_centroid(band: str) -> float:
    """'57-58' -> 57.5  (always two positive integers separated by '-')"""
    if "<" in band:
        return 49.0
    lo, hi = band.split("-")
    return (float(lo) + float(hi)) / 2


def lon_band_centroid(band: str) -> float:
    """'-3 to -2' -> -2.5  (uses ' to ' because '-' is ambiguous with negatives)"""
    if "<" in band:
        return -11.0
    lo, hi = band.split(" to ")
    return (float(lo.strip()) + float(hi.strip())) / 2


# --------------------------------------------------------------------------
# WEATHER FETCHING + CACHING
# --------------------------------------------------------------------------

def fetch_one_chunk(lat, lon, start_date, end_date):
    """Fetch one date range from the API with retry logic."""
    params = {
        "latitude":   lat,
        "longitude":  lon,
        "start_date": start_date,
        "end_date":   end_date,
        "daily":      ",".join(DAILY_VARS),
        "timezone":   "Europe/London",
    }
    error_attempts = 0
    while True:
        try:
            r = requests.get(ARCHIVE_URL, params=params, timeout=90)
            if r.status_code == 200:
                return r.json()
            elif r.status_code == 429:
                print(f"    rate-limited, waiting 65s...")
                time.sleep(65)
            else:
                error_attempts += 1
                if error_attempts >= 6:
                    raise RuntimeError(
                        f"Open-Meteo failed for ({lat},{lon}) {start_date}-{end_date}: "
                        f"{r.status_code} {r.text[:200]}"
                    )
                time.sleep(2 ** error_attempts)
        except requests.exceptions.ReadTimeout:
            error_attempts += 1
            if error_attempts >= 6:
                raise RuntimeError(
                    f"Open-Meteo failed for ({lat},{lon}) {start_date}-{end_date}: timeout"
                )
            print(f"    timeout, retrying ({error_attempts}/6)...")
            time.sleep(10 * error_attempts)

def fetch_location_series(lat: float, lon: float,
                           start_date: str, end_date: str) -> pd.DataFrame:
    """Fetch (or load from cache) a daily weather DataFrame for one location.
    Fetches one year at a time to keep responses small and avoid timeouts."""
    key = f"{lat}_{lon}_{start_date}_{end_date}"
    cache_path = os.path.join(CACHE_DIR, key.replace(" ", "_") + ".parquet")

    if os.path.exists(cache_path):
        return pd.read_parquet(cache_path)

    year_start = int(start_date[:4])
    year_end   = int(end_date[:4])
    chunks = []
    for year in range(year_start, year_end + 1):
        data = fetch_one_chunk(lat, lon, f"{year}-01-01", f"{year}-12-31")
        chunks.append(pd.DataFrame(data["daily"]))
        time.sleep(REQUEST_PAUSE)

    df = pd.concat(chunks, ignore_index=True)
    df["time"] = pd.to_datetime(df["time"])
    df = df.set_index("time")
    df.to_parquet(cache_path)
    return df


# --------------------------------------------------------------------------
# WEATHER FEATURE EXTRACTION PER OBSERVATION
# --------------------------------------------------------------------------

def weather_features(series: pd.DataFrame, event_date: pd.Timestamp) -> dict:
    """
    Compute phenology-relevant weather features from the daily series.

    The scientifically meaningful signals for phenology are the accumulated
    warmth *before* the event, not the weather on the day itself.
    """
    year = event_date.year
    window_start = event_date - pd.Timedelta(days=WINDOW_DAYS)

    window = series.loc[window_start:event_date]
    spring = series.loc[f"{year}-02-01":f"{year}-04-30"]
    winter = series.loc[f"{year-1}-12-01":f"{year}-02-28"]
    full_year = series.loc[f"{year}-01-01":f"{year}-12-31"]

    gdd = (window["temperature_2m_mean"] - GDD_BASE).clip(lower=0).sum()

    # Last frost: latest day before the event where min temp dropped below 0
    frost_before = window[window["temperature_2m_min"] < 0.0]
    last_frost_doy = (
        frost_before.index[-1].dayofyear if not frost_before.empty else 0
    )

    # Spring onset: first day where 7-day rolling mean temp exceeds 5°C
    year_to_event = series.loc[f"{year}-01-01":event_date]
    rolling = year_to_event["temperature_2m_mean"].rolling(7).mean()
    warm_days = rolling[rolling > 5.0]
    spring_onset_doy = int(warm_days.index[0].dayofyear) if not warm_days.empty else 0

    return {
        # --- Look-back window (WINDOW_DAYS before event) ---
        "temp_mean_window":     round(window["temperature_2m_mean"].mean(), 2),
        "temp_max_window":      round(window["temperature_2m_max"].max(), 2),
        "temp_min_window":      round(window["temperature_2m_min"].min(), 2),
        "temp_std_window":      round(window["temperature_2m_mean"].std(), 2),
        "precip_sum_window":    round(window["precipitation_sum"].sum(), 2),
        "dry_days_window":      int((window["precipitation_sum"] < 1.0).sum()),
        "gdd_window":           round(gdd, 2),

        # --- Seasonal aggregates ---
        "temp_mean_spring":     round(spring["temperature_2m_mean"].mean(), 2),
        "temp_mean_winter":     round(winter["temperature_2m_mean"].mean(), 2),
        "precip_sum_spring":    round(spring["precipitation_sum"].sum(), 2),

        # --- Annual summaries ---
        "temp_max_year":        round(full_year["temperature_2m_max"].max(), 2),
        "temp_min_year":        round(full_year["temperature_2m_min"].min(), 2),
        "precip_sum_year":      round(full_year["precipitation_sum"].sum(), 2),
        "frost_days_year":      int((full_year["temperature_2m_min"] < 0.0).sum()),
        "et0_sum_spring":       round(spring["et0_fao_evapotranspiration"].sum(), 2),

        # --- Phenology timing indicators ---
        "last_frost_doy":       last_frost_doy,
        "spring_onset_doy":     spring_onset_doy,
    }


# --------------------------------------------------------------------------
# MAIN ETL
# --------------------------------------------------------------------------

def main():
    print("Loading phenology data...")
    ph = pd.read_csv(PHENOLOGY_CSV, parse_dates=["ObservationDate"])

    ph = ph.dropna(subset=["ObservationDate", "LatitudeBand", "LongitudeBand"])

    ph["band_lat"] = ph["LatitudeBand"].map(lat_band_centroid)
    ph["band_lon"] = ph["LongitudeBand"].map(lon_band_centroid)

    unique_locations = (
        ph[["LatitudeBand", "LongitudeBand", "band_lat", "band_lon"]]
        .drop_duplicates()
        .reset_index(drop=True)
    )
    print(f"  {len(ph):,} observations | "
          f"{len(unique_locations)} unique band cells | "
          f"{ph['Species'].nunique()} species | "
          f"{ph['SpeciesType'].nunique()} species types")

    start_date = f"{ph['Year'].min()}-01-01"
    end_date   = f"{ph['Year'].max()}-12-31"
    print(f"  Date span: {start_date} to {end_date}\n")

    series_by_band = {}
    for i, loc in unique_locations.iterrows():
        label = f"{loc['LatitudeBand']} / {loc['LongitudeBand']}"
        print(f"  [{i+1}/{len(unique_locations)}] fetching ERA5 for {label} "
              f"({loc['band_lat']}, {loc['band_lon']})")
        key = (loc["LatitudeBand"], loc["LongitudeBand"])
        series_by_band[key] = fetch_location_series(
            loc["band_lat"], loc["band_lon"], start_date, end_date
        )

    # Compute weather features for every observation
    print("\nComputing weather features per observation...")
    feats = []
    for row in ph.itertuples(index=False):
        key = (row.LatitudeBand, row.LongitudeBand)
        s   = series_by_band[key]
        feats.append(weather_features(s, row.ObservationDate))

    unified = pd.concat([ph.reset_index(drop=True),
                         pd.DataFrame(feats)], axis=1)

    unified["day_of_year"] = unified["ObservationDate"].dt.dayofyear

    dim_cols  = ["Year", "Month", "Day", "LatitudeBand", "LongitudeBand",
                 "band_lat", "band_lon", "Species", "SpeciesType", "Event"]
    meas_cols = ["day_of_year",
                 # window
                 "temp_mean_window", "temp_max_window", "temp_min_window",
                 "temp_std_window", "precip_sum_window", "dry_days_window",
                 "gdd_window",
                 # seasonal
                 "temp_mean_spring", "temp_mean_winter",
                 "precip_sum_spring", "et0_sum_spring",
                 # annual
                 "temp_max_year", "temp_min_year",
                 "precip_sum_year", "frost_days_year",
                 # timing indicators
                 "last_frost_doy", "spring_onset_doy"]
    id_cols   = ["RecordID", "RecorderID", "ObservationDate",
                 "Latitude", "Longitude"]

    unified = unified[id_cols + dim_cols + meas_cols]
    unified.to_csv(OUTPUT_CSV, index=False)
    print(f"\nWrote {len(unified):,} rows -> {OUTPUT_CSV}")
    print("\nSample row:")
    print(unified.iloc[0].to_string())


if __name__ == "__main__":
    main()

Loading phenology data...
  279,041 observations | 69 unique band cells | 34 species | 5 species types
  Date span: 2014-01-01 to 2024-12-31

  [1/69] fetching ERA5 for 57-58 / -3 to -2 (57.5, -2.5)
  [2/69] fetching ERA5 for 51-52 / 0 to 1 (51.5, 0.5)
  [3/69] fetching ERA5 for 50-51 / -5 to -4 (50.5, -4.5)
  [4/69] fetching ERA5 for 52-53 / -2 to -1 (52.5, -1.5)
  [5/69] fetching ERA5 for 50-51 / -3 to -2 (50.5, -2.5)
  [6/69] fetching ERA5 for 51-52 / -1 to 0 (51.5, -0.5)
  [7/69] fetching ERA5 for 51-52 / -3 to -2 (51.5, -2.5)
  [8/69] fetching ERA5 for 53-54 / -5 to -4 (53.5, -4.5)
  [9/69] fetching ERA5 for 53-54 / -1 to 0 (53.5, -0.5)
  [10/69] fetching ERA5 for 52-53 / 1 to 2 (52.5, 1.5)
  [11/69] fetching ERA5 for 51-52 / -2 to -1 (51.5, -1.5)
  [12/69] fetching ERA5 for 52-53 / -3 to -2 (52.5, -2.5)
  [13/69] fetching ERA5 for 50-51 / 0 to 1 (50.5, 0.5)
  [14/69] fetching ERA5 for 51-52 / 1 to 2 (51.5, 1.5)
  [15/69] fetching ERA5 for 53-54 / 0 to 1 (53.5, 0.5)
  [16/69] fetc

In [6]:
df = pd.read_csv('/content/drive/MyDrive/phenology_weather_cache/phenology_weather_unified.csv')

In [ ]:
df['Event'].unique()

array(['First flowering', 'First recorded', 'First cut', 'Last cut',
       'Recorded all winter', 'First ripe fruit', 'Budburst',
       'First leaf', 'First autumn tinting', 'Full autumn tinting',
       'First leaves falling', 'Bare tree'], dtype=object)

In [ ]:
df.drop({'RecorderID', 'Latitude', 'Longitude',
       'LatitudeBand', 'LongitudeBand'}, axis=1, inplace=True)

In [ ]:
df.rename(columns={"RecordID": "record_id", "ObservationDate": "obs_date", "Species": "species",
           "SpeciesType": "species_type", "Event": "event", "Year": "year",
                   "Month": "month", "Day": "day"}, inplace=True)

In [8]:
# add season column: spring: March, April, May, Summer: June, July, August,
# Autumn: September, October, November, Winter: December, January, February
df['season'] = df['month'].apply(lambda x: 'spring' if x in [3, 4, 5] else ('summer' if x in [6, 7, 8] else ('autumn' if x in [9, 10, 11] else 'winter')))


In [11]:
# save df
df.to_csv('/content/drive/MyDrive/phenology_weather_cache/phenology_weather_unified2.csv', index=False)